# 03 — Mixed Precision Experiment

Compare BF16 vs FP32 training: memory usage, speed, and accuracy.

In [ ]:
import sys
sys.path.insert(0, '../src')

from transformers import AutoTokenizer
from llm_optimization.core import load_config
from llm_optimization.data import load_and_prepare_data, QADataset
from llm_optimization.training import build_mixed_precision_trainer

In [ ]:
config = load_config('../configs/mixed_precision.yaml')
train_df, _, val_df = load_and_prepare_data(config.data)

tokenizer = AutoTokenizer.from_pretrained(config.model_name)
tokenizer.pad_token = tokenizer.eos_token

train_ds = QADataset(train_df, tokenizer, config.data.max_length)
val_ds = QADataset(val_df, tokenizer, config.data.max_length)

trainer, model = build_mixed_precision_trainer(config, train_ds, val_ds, tokenizer)
trainer.train()
trainer.save_model()
tokenizer.save_pretrained(config.output_path)